# Persistent local retrieval benchmark

Build portable exact and compact IVF-PQ artifacts for BioData ProtT5 layer 0. Compare local
exact FAISS CPU, bounded-VRAM exact cuVS streaming, and approximate IVF-PQ retrieval.

## tl;dr

This notebook benchmarks 100 deterministic pseudo-random proteins without running expensive
pgvector nearest-neighbor searches. FAISS CPU exact from the portable exact store is the local
reference. The final cells report recall@10, recall@50, recall@100, and distance RMSD for cuVS
streaming and IVF-PQ relative to that reference.

## Context & Methods

### Key assumptions

- `embedding_type_id=3`, `layer_index=0` identifies the stored ProtT5 protein vectors.
- Cosine distance is the reference metric. IVF-PQ is trained with inner-product search after L2 normalization, so its raw approximate distance is `1 - score`.
- FAISS CPU exact is the local reference. Recall@10, recall@50, and recall@100 compare each method's first K proteins with its exact first K. RMSD compares distances only for shared proteins.
- IVF-PQ builds from the portable float16 exact store. FAISS CPU and cuVS streaming consume that local artifact instead of issuing a full PostgreSQL vector scan.
- PostgreSQL is used only to select deterministic query proteins and build a missing artifact. Once the portable exact store exists, exact FAISS, cuVS streaming, IVF-PQ query vectors, and IVF-PQ candidate IDs are read locally; no search-stage request is sent to PostgreSQL. The notebook does not run pgvector nearest-neighbor search or reranking.


## Setup

In [ ]:
from __future__ import annotations

import gc
import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next((path for path in (Path.cwd(), Path.cwd().parent) if (path / 'CBBIO').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Start Jupyter from the repository root or its notebooks directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from CBBIO import IndexBuildSpec, IndexKey, IndexManager, connect

def _memory_summary() -> tuple[float, float]:
    values = {}
    for line in Path('/proc/meminfo').read_text().splitlines():
        name, value, *_ = line.replace(':', '').split()
        values[name] = int(value) * 1024
    return values['MemAvailable'] / 2**30, values.get('SwapFree', 0) / 2**30


def prepare_local_backend(label: str, *, device: str = 'cuda:0') -> None:
    """Release prior local backend state before a cold backend measurement."""
    client.clear_search_cache()
    gc.collect()
    try:
        import cupy

        device_index = int(device.split(':', maxsplit=1)[1])
        with cupy.cuda.Device(device_index):
            cupy.cuda.runtime.deviceSynchronize()
            cupy.get_default_memory_pool().free_all_blocks()
            cupy.get_default_pinned_memory_pool().free_all_blocks()
            free_bytes, total_bytes = cupy.cuda.runtime.memGetInfo()
        vram_label = f'; VRAM free {free_bytes / 2**30:.1f}/{total_bytes / 2**30:.1f} GiB'
    except Exception:
        vram_label = '; VRAM unavailable'
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass
    available_gib, swap_free_gib = _memory_summary()
    print(f'{label}: cleared in-process search states; RAM available {available_gib:.1f} GiB; '
          f'swap free {swap_free_gib:.1f} GiB{vram_label}')


In [ ]:
# Reproducible experiment parameters. Both config files are local-only and ignored by Git.
BENCHMARK_TARGET = os.environ.get('BIODATA_BENCHMARK_TARGET', 'nas').strip().lower()
if BENCHMARK_TARGET not in {'local', 'nas'}:
    raise ValueError("BIODATA_BENCHMARK_TARGET must be 'local' or 'nas'.")
CONFIG_PATH = PROJECT_ROOT / ('config_test.yaml' if BENCHMARK_TARGET == 'local' else 'config.yaml')
EXPECTED_HOST = 'localhost' if BENCHMARK_TARGET == 'local' else '192.168.49.224'
DATABASE_LABEL = f'biodata-{BENCHMARK_TARGET}'
EMBEDDING_TYPE_ID = 3
LAYER_INDEX = 0
METRIC = 'cosine'
QUERY_COUNT = 500
QUERY_SEED = 20260919991919
TOP_K = 100
RECALL_CUTOFFS = (10, 50, 100)
CANDIDATE_COUNT = 10_000
INDEX_ROOT = Path(os.environ.get('BIODATA_INDEX_ROOT', PROJECT_ROOT / '.biodata' / 'indexes'))

MAX_TRAINING_SAMPLE_SIZE = 124_239
SUBQUANTIZERS = 128
BITS_PER_CODE = 8
NPROBE = 256
CUVS_STREAMING_BLOCK_SIZE = 4_500_000
CUVS_STREAMING_QUERY_BATCH_SIZE = None  # None chooses a conservative automatic limit.
PROFILE_CUVS_STREAMING = True


if max(RECALL_CUTOFFS) > TOP_K:
    raise ValueError('TOP_K must be at least the largest recall cutoff.')

print(f'Benchmark target: {BENCHMARK_TARGET}; config: {CONFIG_PATH.name}; expected host: {EXPECTED_HOST}')
print(f'Index directory: {INDEX_ROOT}')
print(f'Queries: {QUERY_COUNT}; seed: {QUERY_SEED}; top-k: {TOP_K}; candidates: {CANDIDATE_COUNT}')
query_batch_size_label = (
    'auto' if CUVS_STREAMING_QUERY_BATCH_SIZE is None else f'{CUVS_STREAMING_QUERY_BATCH_SIZE:,}'
)
print(f'cuVS streaming block size: {CUVS_STREAMING_BLOCK_SIZE:,} vectors; '
      f'query batch size: {query_batch_size_label}')
print(f'Collect cuVS stage profile: {PROFILE_CUVS_STREAMING}')


## Data

The candidate collection is `sequence_embeddings`, filtered to ProtT5 layer 0. Query proteins come from PostgreSQL `TABLESAMPLE SYSTEM ... REPEATABLE(seed)`, then a seeded hash ordering. This avoids a full-table `count(*)`, scan, or sort when choosing 100 reproducible random queries.

In [ ]:
client = connect(config_path=CONFIG_PATH)
if f'@{EXPECTED_HOST}:' not in client.dsn:
    raise RuntimeError(f'{CONFIG_PATH.name} must target {EXPECTED_HOST}, not a different database.')
server = client.query_one('SELECT current_database() AS database_name, inet_server_addr()::text AS server_address;')
print(f'Connected through {EXPECTED_HOST} to {server["database_name"]} (PostgreSQL address {server["server_address"]}).')


query_rows: list[dict[str, object]] = []
seen_proteins: set[str] = set()
for sample_attempt in range(20):
    if len(query_rows) == QUERY_COUNT:
        break
    table_sample_seed = ((QUERY_SEED + sample_attempt) % 999_983) / 999_983
    rows = client.query_all(
        f'''
        SELECT se.sequence_id, p.id AS protein_id
        FROM sequence_embeddings AS se TABLESAMPLE SYSTEM (0.1) REPEATABLE ({table_sample_seed})
        JOIN protein p ON p.sequence_id = se.sequence_id
        WHERE se.embedding_type_id = %s
          AND se.layer_index = %s
        ORDER BY hashtextextended(se.sequence_id::text, {QUERY_SEED + sample_attempt})
        LIMIT %s;
        ''',
        (EMBEDDING_TYPE_ID, LAYER_INDEX, QUERY_COUNT - len(query_rows)),
    )
    for row in rows:
        protein_id = str(row['protein_id'])
        if protein_id not in seen_proteins:
            query_rows.append({'protein_id': protein_id, 'sequence_id': int(row['sequence_id'])})
            seen_proteins.add(protein_id)

if len(query_rows) < QUERY_COUNT:
    raise RuntimeError('Could not obtain enough random query proteins after 20 deterministic samples.')

query_frame = pd.DataFrame(query_rows)
query_ids = query_frame['protein_id'].tolist()
query_sequence_ids = dict(zip(query_frame['protein_id'], query_frame['sequence_id'], strict=True))
display(query_frame)

## Build persistent retrieval artifacts

This cell first creates or reuses the portable exact float16 store. It is the shared local matrix
for FAISS CPU, cuVS GPU, and IVF-PQ. IVF-PQ is then derived from that store without another full
PostgreSQL vector transfer; both artifacts can be copied to a compute node.

In [ ]:
manager = IndexManager(INDEX_ROOT, search_nprobe=NPROBE)
key = IndexKey.from_biodata(
    client,
    database_label=DATABASE_LABEL,
    embedding_type_id=EMBEDDING_TYPE_ID,
    layer_index=LAYER_INDEX,
    metric=METRIC,
)
revision = client.embedding_index_revision(EMBEDDING_TYPE_ID, LAYER_INDEX)
vector_count = int(revision.split(';', maxsplit=1)[0].split('=', maxsplit=1)[1])
INDEX_SPEC = IndexBuildSpec(
    nlist=min(3476, max(1, int(np.sqrt(vector_count)))),
    subquantizers=SUBQUANTIZERS,
    bits_per_code=BITS_PER_CODE,
    training_sample_size=min(MAX_TRAINING_SAMPLE_SIZE, vector_count),
    nprobe=NPROBE,
)
print(f'Candidate vectors: {vector_count:,}; IVF-PQ nlist: {INDEX_SPEC.nlist:,}; search nprobe: {NPROBE:,}')

exact_inspection = manager.inspect_exact_store(
    database_label=DATABASE_LABEL,
    embedding_type_id=EMBEDDING_TYPE_ID,
    layer_index=LAYER_INDEX,
    source_revision=revision,
)
if exact_inspection.state == 'current':
    exact_build_seconds = 0.0
    exact_manifest = exact_inspection.manifest
    print('Reusing current portable exact store.')
elif exact_inspection.state == 'missing':
    started_at = time.perf_counter()
    exact_manifest = manager.build_exact_store(
        key,
        lambda: client.iter_protein_embedding_index_batches(
            EMBEDDING_TYPE_ID,
            LAYER_INDEX,
            batch_size=10_000,
        ),
        source_revision=revision,
    )
    exact_build_seconds = time.perf_counter() - started_at
    print(f'Built portable exact store with {exact_manifest.vector_count:,} vectors in {exact_build_seconds / 60:.1f} min.')
else:
    raise RuntimeError(
        f'Portable exact store is {exact_inspection.state}: {exact_inspection.reason}. '
        'Rebuild it deliberately with overwrite=True after deciding how to handle the old artifact.'
    )

if exact_manifest is not None:
    print('Portable exact-store build timings:')
    for label, seconds in (
        ('PostgreSQL vector read / transfer', exact_manifest.source_read_seconds),
        ('Save float16 matrix', exact_manifest.vector_write_seconds),
        ('Save SQLite protein-ID metadata', exact_manifest.metadata_write_seconds),
    ):
        if seconds is None:
            print(f'  {label}: unavailable (artifact predates timing instrumentation)')
        else:
            print(f'  {label}: {seconds:.3f} s')

inspection = manager.inspect(key, source_revision=revision)
if inspection.state == 'current':
    build_seconds = 0.0
    print('Reusing current persistent IVF-PQ index.')
elif inspection.state == 'missing':
    started_at = time.perf_counter()
    manifest = manager.build_ivf_pq(
        key,
        lambda: client.iter_protein_embedding_index_batches(
            EMBEDDING_TYPE_ID,
            LAYER_INDEX,
            batch_size=10_000,
        ),
        source_revision=revision,
        spec=INDEX_SPEC,
    )
    build_seconds = time.perf_counter() - started_at
    print(f'Built IVF-PQ from the local exact store with {manifest.vector_count:,} vectors in {build_seconds / 60:.1f} min.')
else:
    raise RuntimeError(
        f'Persistent index is {inspection.state}: {inspection.reason}. '
        'Rebuild it deliberately with overwrite=True after deciding how to handle the old artifact.'
    )

client.configure_persistent_index(manager, database_label=DATABASE_LABEL)
index = manager.load(key, source_revision=revision)
exact_store = manager.load_exact_store(
    database_label=DATABASE_LABEL,
    embedding_type_id=EMBEDDING_TYPE_ID,
    layer_index=LAYER_INDEX,
    source_revision=revision,
)

## Results

cuVS streaming scans the portable exact store with bounded VRAM. FAISS CPU then materializes the
same store and becomes the exact reference for the remaining comparisons. The raw IVF-PQ path
performs approximate local retrieval and one ID-resolution query; it does not invoke pgvector
distance search or reranking.

### Exact cuVS with bounded VRAM

This backend keeps one exact-store block in VRAM at a time. It is exact, but every call scans the
whole local store; it therefore has no warm resident-state measurement. Set
`CUVS_STREAMING_BLOCK_SIZE` to a fixed number of vectors, or leave it as `None` to use a
conservative free-VRAM estimate. It retains candidates through canonical distance ties before
ordering by `(distance, protein_id)`. Stored query vectors and same-sequence exclusions are loaded from the portable exact store, so this search does not query PostgreSQL.


In [ ]:
# 1. Exact cuVS search in bounded GPU memory; every call rescans the portable exact store.
cuvs_streaming_results = None
cuvs_streaming_seconds = None
prepare_local_backend('Before cuVS streaming')
try:
    started_at = time.perf_counter()
    cuvs_streaming_results = client.find_nearest_neighbors_for_proteins(
        query_ids,
        embedding_type_id=EMBEDDING_TYPE_ID,
        layer_index=LAYER_INDEX,
        k=TOP_K,
        metric=METRIC,
        include_query=False,
        backend='cuvs_streaming',
        device='cuda:0',
        use_ann=False,
        cuvs_streaming_block_size=CUVS_STREAMING_BLOCK_SIZE,
        cuvs_streaming_query_batch_size=CUVS_STREAMING_QUERY_BATCH_SIZE,
        profile_cuvs_streaming=PROFILE_CUVS_STREAMING,
    )
    cuvs_streaming_seconds = time.perf_counter() - started_at
    effective_block_size = client.last_search_diagnostics.get('cuvs_streaming_block_size')
    effective_query_batch_size = client.last_search_diagnostics.get('cuvs_streaming_query_batch_size')
    if effective_block_size is None or effective_query_batch_size is None:
        raise RuntimeError('cuVS streaming completed without reporting its effective block and query-batch sizes.')
    effective_query_batch_count = (len(query_ids) + int(effective_query_batch_size) - 1) // int(effective_query_batch_size)
    print(
        f'Exact cuVS streaming: {cuvs_streaming_seconds:.3f} s '
        f'({int(effective_block_size):,} vectors per block; '
        f'{int(effective_query_batch_size):,} queries per batch; '
        f'{effective_query_batch_count:,} query batches)'
    )
    streaming_profile = client.last_search_diagnostics.get('cuvs_streaming_profile')
    if PROFILE_CUVS_STREAMING and streaming_profile is None:
        raise RuntimeError('cuVS profile was requested but no profile was recorded.')
    if streaming_profile is not None:
        profile_rows = [
            (name.removesuffix('_seconds').replace('_', ' '), value)
            for name, value in streaming_profile.items()
            if name.endswith('_seconds') and name != 'total_seconds'
        ]
        profile_frame = pd.DataFrame(profile_rows, columns=['stage', 'seconds'])
        profile_frame['share_of_total'] = profile_frame['seconds'] / streaming_profile['total_seconds']
        profile_frame = profile_frame.sort_values('seconds', ascending=False, ignore_index=True)
        display(profile_frame)
        display(pd.DataFrame([
            {'metric': name, 'value': value}
            for name, value in streaming_profile.items()
            if not name.endswith('_seconds')
        ]))
except Exception as error:
    print(f'Skipping cuVS streaming exact: {type(error).__name__}: {error}')


In [ ]:
# 2. Exact local FAISS CPU reference search from the portable exact store.
# Deliberately clear cuVS pools and all resident states before the FAISS cold measurement.
prepare_local_backend('Before FAISS CPU cold search')
faiss_matrix_gib = exact_manifest.vector_count * exact_manifest.dimension * 4 / 2**30
available_gib, _swap_free_gib = _memory_summary()
if available_gib < faiss_matrix_gib * 1.15:
    print(f'Warning: IndexFlat needs about {faiss_matrix_gib:.1f} GiB plus overhead; '
          f'only {available_gib:.1f} GiB is currently available. The cold load may swap.')
started_at = time.perf_counter()
faiss_cpu_results = client.find_nearest_neighbors_for_proteins(
    query_ids,
    embedding_type_id=EMBEDDING_TYPE_ID,
    layer_index=LAYER_INDEX,
    k=TOP_K,
    metric=METRIC,
    include_query=False,
    backend='faiss_cpu',
    use_ann=False,
)
faiss_cpu_seconds = time.perf_counter() - started_at
print(f'Exact FAISS CPU: {faiss_cpu_seconds:.3f} s')


### Cold-start versus warm exact local search

The first exact call materializes the backend state from the local portable exact store (and, for cuVS, transfers/prepares it on the GPU). The repeated call uses that resident state. Both calls still fetch the query vectors and format their results, so `cold − warm` is an estimate of state loading/materialization rather than an isolated I/O measurement.

In [ ]:
# 2b. Repeat the FAISS CPU reference with the exact-store-derived index state already resident.
started_at = time.perf_counter()
faiss_cpu_warm_results = client.find_nearest_neighbors_for_proteins(
    query_ids,
    embedding_type_id=EMBEDDING_TYPE_ID,
    layer_index=LAYER_INDEX,
    k=TOP_K,
    metric=METRIC,
    include_query=False,
    backend='faiss_cpu',
    use_ann=False,
)
faiss_cpu_warm_seconds = time.perf_counter() - started_at
faiss_cpu_load_estimate_seconds = max(0.0, faiss_cpu_seconds - faiss_cpu_warm_seconds)
print(f'FAISS CPU exact, warm: {faiss_cpu_warm_seconds:.3f} s')
print(f'FAISS CPU state loading/materialization (estimated): {faiss_cpu_load_estimate_seconds:.3f} s')

### Profile warm FAISS CPU

Run this optional diagnostic if the warm FAISS timing is surprising. It primes the resident exact
state, then profiles one additional public API call. This separates FAISS engine time from Python,
query-vector loading, and result handling. The profile is excluded from `timing_frame`.

In [ ]:
# Optional: this primes the selected state, then profiles one additional warm exact batch.
PROFILE_EXACT_BACKEND = None

if PROFILE_EXACT_BACKEND is not None:
    import cProfile
    import io
    import pstats

    if PROFILE_EXACT_BACKEND != 'faiss_cpu':
        raise ValueError("PROFILE_EXACT_BACKEND must be 'faiss_cpu' or None.")

    profile_kwargs = {
        'embedding_type_id': EMBEDDING_TYPE_ID,
        'layer_index': LAYER_INDEX,
        'k': TOP_K,
        'metric': METRIC,
        'include_query': False,
        'backend': PROFILE_EXACT_BACKEND,
        'use_ann': False,
    }
    # The cache retains one state per compute resource, so explicitly prime the selected backend.
    _ = client.find_nearest_neighbors_for_proteins(query_ids, **profile_kwargs)

    profiler = cProfile.Profile()
    started_at = time.perf_counter()
    profiler.enable()
    _ = client.find_nearest_neighbors_for_proteins(query_ids, **profile_kwargs)
    profiler.disable()
    print(f'{PROFILE_EXACT_BACKEND} warm API call: {time.perf_counter() - started_at:.3f} s')

    profile_output = io.StringIO()
    pstats.Stats(profiler, stream=profile_output).strip_dirs().sort_stats('cumulative').print_stats(20)
    print(profile_output.getvalue())
else:
    print("Skipped. Set PROFILE_EXACT_BACKEND to 'faiss_cpu' to profile one warm batch.")

In [ ]:
# 3. IVF-PQ only: scores are approximate cosine similarities, with no distance reranking.
# Release the resident FAISS IndexFlat; the compact IVF-PQ index remains referenced by `index`.
prepare_local_backend('Before IVF-PQ search')
# Resolve the full candidate pool before selecting TOP_K distinct protein IDs.
# Use the same float16-derived query vectors as the local exact backends; no PostgreSQL read.
portable_query_vectors, _portable_query_aliases, _portable_query_sequence_ids = manager.get_exact_store_query_context(
    exact_inspection,
    query_ids,
)
if len(portable_query_vectors) != len(query_ids):
    missing_query_ids = sorted(set(query_ids) - set(portable_query_vectors))
    raise RuntimeError(f'Portable exact store is missing selected query proteins: {missing_query_ids[:5]}')
query_vectors = portable_query_vectors
started_at = time.perf_counter()
raw_candidate_pools = {
    query_id: [
        candidate
        for candidate in manager.search(
            index,
            query_vectors[query_id],
            key=key,
            candidate_count=CANDIDATE_COUNT,
        )
        if candidate.sequence_id != query_sequence_ids[query_id]
    ]
    for query_id in query_ids
}
raw_seconds = time.perf_counter() - started_at

candidate_sequence_ids = sorted(
    {
        candidate.sequence_id
        for candidates in raw_candidate_pools.values()
        for candidate in candidates
    }
)
protein_ids_by_sequence_id = manager.get_exact_store_protein_ids_by_sequence_id(
    exact_inspection,
    candidate_sequence_ids,
)
protein_by_sequence_id = {
    sequence_id: protein_ids[0]
    for sequence_id, protein_ids in protein_ids_by_sequence_id.items()
    if protein_ids
}
raw_results = {}
for query_id, candidates in raw_candidate_pools.items():
    seen_protein_ids = set()
    neighbors = []
    for candidate in candidates:
        protein_id = protein_by_sequence_id.get(candidate.sequence_id)
        if protein_id is None or protein_id in seen_protein_ids:
            continue
        seen_protein_ids.add(protein_id)
        neighbors.append(
            {
                'protein_id': protein_id,
                'distance': 1.0 - candidate.score,
            }
        )
        if len(neighbors) == TOP_K:
            break
    raw_results[query_id] = neighbors

print(f'Raw IVF-PQ: {raw_seconds:.3f} s (plus one ID-resolution query)')


In [ ]:
def _neighbors_to_distances(neighbors):
    return {neighbor.protein_id: float(neighbor.distance) for neighbor in neighbors}


def _result_protein_ids(method_name, results):
    if method_name == 'IVF-PQ only':
        return [item['protein_id'] for item in results]
    return [neighbor.protein_id for neighbor in results]


def _evaluate_method(method_name, results, reference_results):
    rows = []
    for query_id in query_ids:
        reference_protein_ids = [neighbor.protein_id for neighbor in reference_results[query_id]]
        method_protein_ids = _result_protein_ids(method_name, results[query_id])
        reference_by_protein = _neighbors_to_distances(reference_results[query_id])
        if method_name == 'IVF-PQ only':
            method_by_protein = {item['protein_id']: float(item['distance']) for item in results[query_id]}
        else:
            method_by_protein = _neighbors_to_distances(results[query_id])
        shared_proteins = sorted(set(reference_by_protein) & set(method_by_protein))
        squared_errors = [
            (method_by_protein[protein_id] - reference_by_protein[protein_id]) ** 2
            for protein_id in shared_proteins
        ]
        row = {
            'method': method_name,
            'query_id': query_id,
            'shared_neighbors': len(shared_proteins),
            'rmsd_distance': float(np.sqrt(np.mean(squared_errors))) if squared_errors else np.nan,
        }
        for cutoff in RECALL_CUTOFFS:
            reference_at_cutoff = set(reference_protein_ids[:cutoff])
            method_at_cutoff = set(method_protein_ids[:cutoff])
            row[f'recall_at_{cutoff}'] = len(reference_at_cutoff & method_at_cutoff) / cutoff
        rows.append(row)
    return rows


METHOD_COLORS = {
    'FAISS CPU exact': '#4C78A8',
    'cuVS streaming exact': '#76B7B2',
    'IVF-PQ only': '#B07AA1',
}
reference_results = faiss_cpu_results
method_results = [
    ('FAISS CPU exact', reference_results),
    ('cuVS streaming exact', globals().get('cuvs_streaming_results')),
    ('IVF-PQ only', globals().get('raw_results')),
]
available_method_results = [
    (method_name, results)
    for method_name, results in method_results
    if results is not None
]
skipped_method_names = [
    method_name
    for method_name, results in method_results
    if results is None
]
print('Reference: FAISS CPU exact')
print('Evaluated methods: ' + ', '.join(method_name for method_name, _ in available_method_results))
if skipped_method_names:
    print('Skipped or unavailable: ' + ', '.join(skipped_method_names))

quality_rows = [
    row
    for method_name, results in available_method_results
    for row in _evaluate_method(method_name, results, reference_results)
]
quality_frame = pd.DataFrame(quality_rows)

timing_candidates = [
    ('FAISS CPU exact (cold)', globals().get('faiss_cpu_seconds')),
    ('FAISS CPU exact (warm)', globals().get('faiss_cpu_warm_seconds')),
    ('FAISS CPU state loading/materialization (estimated)', globals().get('faiss_cpu_load_estimate_seconds')),
    ('cuVS streaming exact (full store scan)', globals().get('cuvs_streaming_seconds')),
    ('IVF-PQ only', globals().get('raw_seconds')),
]
non_search_timing_methods = {
    'FAISS CPU state loading/materialization (estimated)',
}
timing_frame = pd.DataFrame(
    {
        'method': method_name,
        'elapsed_seconds': seconds,
        'queries_per_second': (
            len(query_ids) / seconds
            if method_name not in non_search_timing_methods and seconds > 0
            else np.nan
        ),
    }
    for method_name, seconds in timing_candidates
    if seconds is not None
)
summary_frame = quality_frame.groupby('method', as_index=False).agg(
    mean_recall_at_10=('recall_at_10', 'mean'),
    min_recall_at_10=('recall_at_10', 'min'),
    mean_recall_at_50=('recall_at_50', 'mean'),
    min_recall_at_50=('recall_at_50', 'min'),
    mean_recall_at_100=('recall_at_100', 'mean'),
    min_recall_at_100=('recall_at_100', 'min'),
    mean_rmsd_distance=('rmsd_distance', 'mean'),
)
display(timing_frame)
display(summary_frame)
display(quality_frame.sort_values(['method', 'recall_at_100', 'query_id']))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
recall_columns = ['mean_recall_at_10', 'mean_recall_at_50', 'mean_recall_at_100']
recall_labels = ['recall@10', 'recall@50', 'recall@100']
positions = np.arange(len(recall_columns))
bar_width = 0.16
for method_index, (_, method_row) in enumerate(summary_frame.iterrows()):
    axes[0].bar(
        positions + (method_index - (len(summary_frame) - 1) / 2) * bar_width,
        [method_row[column] for column in recall_columns],
        width=bar_width,
        color=METHOD_COLORS[method_row['method']],
        label=method_row['method'],
    )
axes[0].set_xticks(positions, recall_labels)
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('Mean recall')
axes[0].set_title('Recovery against FAISS CPU exact')
axes[0].legend()

axes[1].bar(
    summary_frame['method'],
    summary_frame['mean_rmsd_distance'],
    color=[METHOD_COLORS[method] for method in summary_frame['method']],
)
axes[1].set_ylabel('Mean RMSD of shared distances')
axes[1].set_title('Distance fidelity')
axes[1].tick_params(axis='x', rotation=15)

fig.tight_layout()
plt.show()

### Per-query recall

Each point is one query protein. Use the sorted per-query table above to identify its sequence ID.

In [ ]:
method_names = summary_frame['method'].tolist()
recall_distributions = [
    quality_frame.loc[quality_frame['method'] == method_name, 'recall_at_100'].to_numpy()
    for method_name in method_names
]
palette = [METHOD_COLORS[method_name] for method_name in method_names]

fig, axis = plt.subplots(figsize=(11, 4))
boxplot = axis.boxplot(
    recall_distributions,
    tick_labels=method_names,
    patch_artist=True,
    showmeans=True,
    medianprops={'color': '#222222'},
    meanprops={'marker': 'D', 'markerfacecolor': '#222222', 'markeredgecolor': '#222222'},
)
for patch, color in zip(boxplot['boxes'], palette, strict=True):
    patch.set_facecolor(color)
    patch.set_alpha(0.35)
for position, values in enumerate(recall_distributions, start=1):
    offsets = np.linspace(-0.08, 0.08, num=len(values))
    axis.scatter(position + offsets, values, color='#222222', s=28, zorder=3)

axis.set_ylim(0, 1.02)
axis.set_ylabel('recall@100 per query protein')
axis.set_title('Tail recall across query proteins')
axis.tick_params(axis='x', rotation=25)
axis.grid(axis='y', color='#DDDDDD', linewidth=0.8)
fig.tight_layout()
plt.show()

## Takeaways

Interpret the executed table as follows:

- FAISS CPU exact is the local reference. Its cold time materializes the `IndexFlat` state from the portable exact store. Its warm time measures repeated exact batch search.
- cuVS streaming should recover the same neighbors as FAISS while using bounded VRAM. Its time includes one full local-store scan for the whole 100-query batch.
- IVF-PQ-only recall@10, recall@50, and recall@100 measure local approximate-retrieval quality. Its distance RMSD reflects PQ approximation error.
- Increase `CANDIDATE_COUNT` or `nprobe` if approximate recall@K is insufficient. Record the associated local-search latency.
- This notebook deliberately omits pgvector nearest-neighbor search and exact database reranking. It measures local scaling without the remote database baseline cost.